In [1]:
from pathlib import Path

LAST_NAME = "PARRAS"
STUDENT_ID = "TUPM-26-3597"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGHT =len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID sum:", ID_SUM)
print("Name Lenght:", NAME_LENGHT)





Seed: 7
ID sum: 32
Name Lenght: 6


In [4]:

# Import tools for reading, writing, and backing up files.
import csv
import json
import shutil
from pathlib import Path

# Set the location of the original dataset.
input_file = Path(r"C:\Users\JULIANNA\Downloads\Global_Cybersecurity_Threats_2015-2024.csv")


# Calculate the target year using the worksheet's rule.
target_year = 2015 + SEED_DIGIT

# Set the names of the output files.
filtered_file = Path.cwd() / "filtered_incidents.csv"
summary_file = Path.cwd() / "incident_summary.json"
backup_file = Path.cwd() / "incident_summary_backup.json"
report_file = Path.cwd() / "incident_report.txt"

# Display the assigned target year.
print("Target year:", target_year)


#===============================================================================



# Open the original dataset.
with open(input_file, "r", newline="", encoding="utf-8-sig") as file:

    # Read the CSV column names and records.
    reader = csv.DictReader(file)

    # Store the column names.
    headers = reader.fieldnames

    # Remove extra spaces from the column names.
    headers = [name.strip() for name in headers]

    # Store the dataset records.
    data = list(reader)

# Display the dataset information.
print("Total records in dataset:", len(data))
print("CSV columns:", headers)



Target year: 2015
Total records in dataset: 3000
CSV columns: ['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']


In [6]:

# Prepare a list for incidents from the target year.
selected_incidents = []

# Check each record in the dataset.
for row in data:

    # Read the year and convert it to an integer.
    year = int(row["Year"])

    # Keep only incidents from the assigned year.
    if year == target_year:
        selected_incidents.append(row)

# Count the selected incidents.
incident_count = len(selected_incidents)

# Display the number of matching incidents.
print("Target year:", target_year)
print("Incidents in target year:", incident_count)

#===============================================================================


# Prepare a dictionary to count each attack type.
attack_counts = {}

# Store valid financial loss values for calculating averages.
loss_values = []

# Check every incident from the target year.
for row in selected_incidents:

    # Read the attack type.
    attack_type = row["Attack Type"].strip()

    # Add one to the count for this attack type.
    attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1

    # Read the financial loss in millions of dollars.
    loss = float(row["Financial Loss (in Million $)"])

    # Store the financial loss.
    loss_values.append(loss)

# Calculate the total and average financial loss.
total_loss = sum(loss_values)

# Avoid dividing by zero if there are no incidents.
if len(loss_values) > 0:
    average_loss = total_loss / len(loss_values)
else:
    average_loss = None

# Display the results.
print("Count per attack type:")
for attack_type, count in sorted(attack_counts.items()):
    print(attack_type, ":", count)

print("Total financial loss (USD millions):", total_loss)
print("Average financial loss (USD millions):", average_loss)



Target year: 2015
Incidents in target year: 277
Count per attack type:
DDoS : 50
Malware : 51
Man-in-the-Middle : 41
Phishing : 46
Ransomware : 47
SQL Injection : 42
Total financial loss (USD millions): 14510.21
Average financial loss (USD millions): 52.383429602888086


In [8]:

# Sort incidents by financial loss, from highest to lowest.
# Each row's loss is converted to a number for comparison.
costliest_incidents = sorted(
    selected_incidents,
    key=lambda row: float(row["Financial Loss (in Million $)"]),
    reverse=True
)

# Keep only the five incidents with the highest financial losses.
top_five = costliest_incidents[:5]

# Display the five costliest incidents.
print("Five costliest incidents:")

for incident in top_five:
    print(
        "Year:", incident["Year"],
        "| Attack:", incident["Attack Type"],
        "| Loss (USD millions):", incident["Financial Loss (in Million $)"]
    )



#===============================================================================


# Use all the original columns for the filtered CSV.
export_fields = headers

# Open a new CSV file for writing.
with open(filtered_file, "w", newline="", encoding="utf-8") as file:

    # Create the CSV writer.
    writer = csv.DictWriter(file, fieldnames=export_fields)

    # Write the column headers.
    writer.writeheader()

    # Write all incidents from the target year.
    writer.writerows(selected_incidents)

# Display the export results.
print("Filtered CSV file:", filtered_file)
print("Incidents exported:", incident_count)


Five costliest incidents:
Year: 2015 | Attack: Ransomware | Loss (USD millions): 99.83
Year: 2015 | Attack: Malware | Loss (USD millions): 99.41
Year: 2015 | Attack: Phishing | Loss (USD millions): 99.2
Year: 2015 | Attack: SQL Injection | Loss (USD millions): 98.78
Year: 2015 | Attack: Man-in-the-Middle | Loss (USD millions): 98.58
Filtered CSV file: c:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab6_File_HAndling_Parras\filtered_incidents.csv
Incidents exported: 277


In [ ]:

# Prepare the summary using the analysis results.
summary = {
    "target_year": target_year,
    "total_records_in_dataset": len(data),
    "incidents_in_target_year": incident_count,
    "attack_type_counts": attack_counts,
    "total_financial_loss_usd_millions": total_loss,
    "average_financial_loss_usd_millions": average_loss,
    "five_costliest_incidents": top_five,
    "filtered_csv_file": filtered_file.name
}

# Save the summary in a JSON file.
with open(summary_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# Create a backup of the JSON summary.
shutil.copy2(summary_file, backup_file)

# Read the original summary.
with open(summary_file, "r", encoding="utf-8") as file:
    original_summary = json.load(file)

# Read the backup summary.
with open(backup_file, "r", encoding="utf-8") as file:
    backup_summary = json.load(file)

# Verify that the backup contains the same data.
backup_verified = original_summary == backup_summary

# Display the summary and backup verification.
print("JSON summary file:", summary_file)
print(json.dumps(summary, indent=4))
print("Backup file:", backup_file)
print("Backup verification:", backup_verified)

#===============================================================================




JSON summary file: c:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab6_File_HAndling_Parras\incident_summary.json
{
    "target_year": 2015,
    "total_records_in_dataset": 3000,
    "incidents_in_target_year": 277,
    "attack_type_counts": {
        "DDoS": 50,
        "Ransomware": 47,
        "SQL Injection": 42,
        "Phishing": 46,
        "Malware": 51,
        "Man-in-the-Middle": 41
    },
    "total_financial_loss_usd_millions": 14510.21,
    "average_financial_loss_usd_millions": 52.383429602888086,
    "five_costliest_incidents": [
        {
            "Country": "Germany",
            "Year": "2015",
            "Attack Type": "Ransomware",
            "Target Industry": "Telecommunications",
            "Financial Loss (in Million $)": "99.83",
            "Number of Affected Users": "419344",
            "Attack Source": "Hacker Group",
            "Security Vulnerability Type": "Zero-day",
            "Defense Mechanism Used": "Firewall",
            "Incident Resol

In [10]:

# Prepare the report heading and basic incident information.
report_lines = [
    "CYBERSECURITY INCIDENT REPORT",
    "Target year: " + str(target_year),
    "Total incidents in target year: " + str(incident_count),
    "Total financial loss (USD millions): " + str(total_loss),
    "Average financial loss (USD millions): " + str(average_loss),
    "",
    "COUNT PER ATTACK TYPE:"
]

# Add each attack type and its count to the report.
for attack_type, count in sorted(attack_counts.items()):
    report_lines.append(attack_type + ": " + str(count))

# Add the five costliest incidents to the report.
report_lines.append("")
report_lines.append("FIVE COSTLIEST INCIDENTS:")

for incident in top_five:
    report_lines.append(
        "Year: " + str(incident["Year"]) +
        " | Attack: " + incident["Attack Type"] +
        " | Loss (USD millions): " +
        str(incident["Financial Loss (in Million $)"])
    )

# Save the report as a text file.
with open(report_file, "w", encoding="utf-8") as file:
    file.write("\n".join(report_lines))

# Display the report location and contents.
print("Report file:", report_file)
print("\n".join(report_lines))


Report file: c:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab6_File_HAndling_Parras\incident_report.txt
CYBERSECURITY INCIDENT REPORT
Target year: 2015
Total incidents in target year: 277
Total financial loss (USD millions): 14510.21
Average financial loss (USD millions): 52.383429602888086

COUNT PER ATTACK TYPE:
DDoS: 50
Malware: 51
Man-in-the-Middle: 41
Phishing: 46
Ransomware: 47
SQL Injection: 42

FIVE COSTLIEST INCIDENTS:
Year: 2015 | Attack: Ransomware | Loss (USD millions): 99.83
Year: 2015 | Attack: Malware | Loss (USD millions): 99.41
Year: 2015 | Attack: Phishing | Loss (USD millions): 99.2
Year: 2015 | Attack: SQL Injection | Loss (USD millions): 98.78
Year: 2015 | Attack: Man-in-the-Middle | Loss (USD millions): 98.58


In [11]:

# Display every assessment item from the worksheet.
print("========== ASSESSMENT DATA ==========")

print("Target year:", target_year)
print("Total records in dataset:", len(data))
print("Incidents in target year:", incident_count)

print("\nCount per attack type:")
for attack_type, count in sorted(attack_counts.items()):
    print(attack_type, ":", count)

print("\nTotal financial loss (USD millions):", total_loss)
print("Average financial loss (USD millions):", average_loss)

print("\nFive costliest incidents:")
for incident in top_five:
    print(
        incident["Attack Type"],
        "- USD millions:",
        incident["Financial Loss (in Million $)"]
    )

print("\nFiltered CSV file:", filtered_file.name)
print("JSON summary file:", summary_file.name)
print("Backup file:", backup_file.name)
print("Backup verification:", backup_verified)
print("Report file:", report_file.name)

print("=====================================")


========== ASSESSMENT DATA ==========
Target year: 2015
Total records in dataset: 3000
Incidents in target year: 277

Count per attack type:
DDoS : 50
Malware : 51
Man-in-the-Middle : 41
Phishing : 46
Ransomware : 47
SQL Injection : 42

Total financial loss (USD millions): 14510.21
Average financial loss (USD millions): 52.383429602888086

Five costliest incidents:
Ransomware - USD millions: 99.83
Malware - USD millions: 99.41
Phishing - USD millions: 99.2
SQL Injection - USD millions: 98.78
Man-in-the-Middle - USD millions: 98.58

Filtered CSV file: filtered_incidents.csv
JSON summary file: incident_summary.json
Backup file: incident_summary_backup.json
Backup verification: True
Report file: incident_report.txt
